In [2]:
import glob

import numpy as np
import pandas as pd
from model.ctabgan import CTABGAN
from model.eval.evaluation import get_utility_metrics, privacy_metrics, stat_sim

ImportError: cannot import name 'compute_associations' from 'dython.nominal' (/workspace/tmp_micromamba/root/envs/thesis310/lib/python3.10/site-packages/dython/nominal.py)

In [3]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.7.0+cu128
True
1
NVIDIA H100 PCIe


In [4]:
num_exp = 5
dataset = "MimicIV"
real_path = "Real_Datasets/MimicIV_clean.csv"
fake_file_root = "Fake_Datasets"

In [5]:
all_discriminator_snapshots = {}

for i in range(num_exp):

    print(f"Training run {i}")
    
    synthesizer = CTABGAN(
        raw_csv_path=real_path,
        test_ratio=0.20,
        categorical_columns=["gender", "hospital_expire_flag"],
        log_columns=[],  # none unless you want to log-transform skewed labs
        mixed_columns={},  # none unless you confirm heavy zero-mass columns
        general_columns=[
            "heart_rate_min",
            "heart_rate_max",
            "heart_rate_mean",
            "sbp_min",
            "sbp_max",
            "sbp_mean",
            "dbp_min",
            "dbp_max",
            "dbp_mean",
            "resp_rate_min",
            "resp_rate_max",
            "resp_rate_mean",
            "temperature_min",
            "temperature_max",
            "temperature_mean",
            "spo2_min",
            "spo2_max",
            "spo2_mean",
            "creatinine_min",
            "creatinine_max",
            "sodium_min",
            "sodium_max",
            "potassium_min",
            "potassium_max",
            "hemoglobin_min",
            "hemoglobin_max",
            "wbc_min",
            "wbc_max",
        ],
        non_categorical_columns=[],  # since we dropped IDs
        integer_columns=["age_at_intime"],
        problem_type={"Classification": "hospital_expire_flag"},
        class_dim=(256, 256),
        random_dim=100,
        num_channels=64,
        l2scale=1e-5,
        batch_size=512,
        epochs=150,
    )

    discriminator_snapshots = synthesizer.fit()

    all_discriminator_snapshots[f"run_{i}"] = discriminator_snapshots
    
    syn = synthesizer.generate_samples()
    syn.to_csv(
        fake_file_root
        + "/"
        + dataset
        + "/"
        + dataset
        + "_fake_{exp}.csv".format(exp=i),
        index=False,
    )

dside = synthesizer.synthesizer.dside
layers_D = synthesizer.synthesizer.layers_D

Training run 0


100%|██████████| 150/150 [08:10<00:00,  3.27s/it]


Finished training in 494.59265422821045  seconds.
Training run 1


100%|██████████| 150/150 [08:15<00:00,  3.30s/it]


Finished training in 498.0100984573364  seconds.
Training run 2


...

Finished training in 498.8529815673828  seconds.
Training run 3


100%|██████████| 150/150 [08:23<00:00,  3.36s/it]


Finished training in 506.2873749732971  seconds.
Training run 4


100%|██████████| 150/150 [09:31<00:00,  3.81s/it]


Finished training in 574.3929941654205  seconds.


In [6]:
fake_paths = glob.glob(fake_file_root + "/" + dataset + "/" + "*")

In [7]:
def make_numeric_for_utility(path_in, path_out, target="hospital_expire_flag"):
    df = pd.read_csv(path_in)

    # Ensure target is last or separated (depending on your eval code)
    y = df[target]
    X = df.drop(columns=[target])

    # One-hot encode any object/categorical columns (e.g., gender)
    X_enc = pd.get_dummies(X, drop_first=False)

    out = pd.concat([X_enc, y], axis=1)
    out.to_csv(path_out, index=False)
    return path_out


real_path_u = make_numeric_for_utility(real_path, "Real_Datasets/MimicIVU.csv")

fake_paths_u = []
for i, fp in enumerate(fake_paths):
    fake_paths_u.append(
        make_numeric_for_utility(fp, f"Fake_Datasets/MimicIV/MimicIV_fakeU_{i}.csv")
    )

    # after generating real one-hot columns, align fake to real:
real_cols = pd.read_csv(real_path_u).columns


def align_to_real(path_in, real_cols, target):
    df = pd.read_csv(path_in)
    y = df[target]
    X = pd.get_dummies(df.drop(columns=[target]), drop_first=False)

    X = X.reindex(columns=[c for c in real_cols if c != target], fill_value=0)
    out = pd.concat([X, y], axis=1)
    return out

In [8]:
import pandas as pd

df = pd.read_csv(fake_file_root + "/" + dataset + "/" + dataset + "_fakeU_0.csv")
df.info()
df.head()
df.tail()
df.isna().sum()
df.replace([np.inf, -np.inf], np.nan).isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23698 entries, 0 to 23697
Data columns (total 32 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   age_at_intime         23698 non-null  int64  
 1   heart_rate_min        23698 non-null  float64
 2   heart_rate_max        23698 non-null  float64
 3   heart_rate_mean       23698 non-null  float64
 4   sbp_min               23698 non-null  float64
 5   sbp_max               23698 non-null  float64
 6   sbp_mean              23698 non-null  float64
 7   dbp_min               23698 non-null  float64
 8   dbp_max               23698 non-null  float64
 9   dbp_mean              23698 non-null  float64
 10  resp_rate_min         23698 non-null  float64
 11  resp_rate_max         23698 non-null  float64
 12  resp_rate_mean        23698 non-null  float64
 13  temperature_min       23698 non-null  float64
 14  temperature_max       23698 non-null  float64
 15  temperature_mean   

age_at_intime           0
heart_rate_min          0
heart_rate_max          0
heart_rate_mean         0
sbp_min                 0
sbp_max                 0
sbp_mean                0
dbp_min                 0
dbp_max                 0
dbp_mean                0
resp_rate_min           0
resp_rate_max           0
resp_rate_mean          0
temperature_min         0
temperature_max         0
temperature_mean        0
spo2_min                0
spo2_max                0
spo2_mean               0
creatinine_min          0
creatinine_max          0
sodium_min              0
sodium_max              0
potassium_min           0
potassium_max           0
hemoglobin_min          0
hemoglobin_max          0
wbc_min                 0
wbc_max                 0
gender_F                0
gender_M                0
hospital_expire_flag    0
dtype: int64

In [9]:
model_dict = {"Classification": ["lr", "dt", "rf", "mlp", "svm"]}
result_mat = get_utility_metrics(
    real_path, fake_paths, "MinMax", model_dict, test_ratio=0.20
)

result_df = pd.DataFrame(result_mat, columns=["Acc", "AUC", "F1_Score"])
result_df.index = list(model_dict.values())[0]
result_df

NameError: name 'get_utility_metrics' is not defined

In [10]:
mimic_categorical = ["hospital_expire_flag", "gender"]
stat_res_avg = []
for fake_path in fake_paths:
    stat_res = stat_sim(real_path, fake_path, mimic_categorical)
    stat_res_avg.append(stat_res)

stat_columns = [
    "Average WD (Continuous Columns",
    "Average JSD (Categorical Columns)",
    "Correlation Distance",
]
stat_results = pd.DataFrame(
    np.array(stat_res_avg).mean(axis=0).reshape(1, 3), columns=stat_columns
)
stat_results

NameError: name 'stat_sim' is not defined

In [11]:
priv_res_avg = []
for fake_path in fake_paths_u:
    priv_res = privacy_metrics(real_path_u, fake_path)
    priv_res_avg.append(priv_res)

privacy_columns = [
    "DCR between Real and Fake (5th perc)",
    "DCR within Real(5th perc)",
    "DCR within Fake (5th perc)",
    "NNDR between Real and Fake (5th perc)",
    "NNDR within Real (5th perc)",
    "NNDR within Fake (5th perc)",
]
privacy_results = pd.DataFrame(
    np.array(priv_res_avg).mean(axis=0).reshape(1, 6), columns=privacy_columns
)
privacy_results

NameError: name 'privacy_metrics' is not defined

In [12]:
import os

import pandas as pd
from Evaluation.cdf_tail_metrics import CDFTailMetrics
from Evaluation.rare_event_recall import RareEventRecall
from Evaluation.support_coverage import SupportCoverage

n_experiments = 5

cdf_eval = CDFTailMetrics(label_col="hospital_expire_flag", tau=0.9)
sc_eval = SupportCoverage(
    label_col="hospital_expire_flag",
    k=2,  # pairwise support coverage
    n_bins=10,
    rare_threshold=0.05,
    max_subsets=100,  # sample up to 100 feature pairs
    include_label_in_combo=True,
    random_state=42,
)
rer_eval = RareEventRecall(label_col="hospital_expire_flag")

rows = []

for exp in range(n_experiments):
    syn_path = os.path.join(fake_file_root, dataset, f"{dataset}_fakeU_{exp}.csv")
    if not os.path.exists(syn_path):
        print("Missing:", syn_path)
        continue

    res = {"exp": exp, "syn_path": syn_path}
    res.update(cdf_eval.evaluate_paths(real_path_u, syn_path))
    res.update(sc_eval.evaluate_paths(real_path_u, syn_path))
    res.update(rer_eval.evaluate_paths(real_path_u, syn_path))
    rows.append(res)

results_df = pd.DataFrame(rows)
results_df.to_csv("artifacts/MimicEval.csv")
results_df

,exp,syn_path,cdf_tail_div_age_at_intime,cdf_tail_div_heart_rate_min,cdf_tail_div_heart_rate_max,cdf_tail_div_heart_rate_mean,cdf_tail_div_sbp_min,cdf_tail_div_sbp_max,cdf_tail_div_sbp_mean,cdf_tail_div_dbp_min,...,avg_num_syn_combos,rare_class,rare_event_recall_real_baseline,rare_event_total_real_baseline,rare_event_correct_real_baseline,rare_event_recall_synthetic,rare_event_total_synthetic,rare_event_correct_synthetic,rare_class_count_real,n_classes_real
0,0,Fake_Datasets/MimicIV/MimicIV_fakeU_0.csv,0.010465,0.052705,0.049962,0.004811,0.015824,0.025741,0.012744,0.022069,...,174.93,1.0,0.219844,1028,226,0.303358,3425,1039,3425,2
1,1,Fake_Datasets/MimicIV/MimicIV_fakeU_1.csv,0.010085,0.051988,0.015149,0.013967,0.020635,0.012491,0.018651,0.009959,...,176.57,1.0,0.219844,1028,226,0.184526,3425,632,3425,2
2,2,Fake_Datasets/MimicIV/MimicIV_fakeU_2.csv,0.013334,0.014769,0.025825,0.002532,0.019833,0.027217,0.009832,0.015065,...,176.94,1.0,0.219844,1028,226,0.262774,3425,900,3425,2
3,3,Fake_Datasets/MimicIV/MimicIV_fakeU_3.csv,0.010085,0.051988,0.015149,0.013967,0.020635,0.012491,0.018651,0.009959,...,176.57,1.0,0.219844,1028,226,0.184526,3425,632,3425,2
4,4,Fake_Datasets/MimicIV/MimicIV_fakeU_4.csv,0.012406,0.022702,0.049287,0.045531,0.062453,0.039708,0.032070,0.027260,...,175.28,1.0,0.219844,1028,226,0.246423,3425,844,3425,2


In [13]:
from Evaluation.detector_treeshap import DetectorTreeSHAP

xai_detector = DetectorTreeSHAP(
    test_size=0.3,
    random_state=42,
    n_estimators=300,
    output_dir=f"artifacts/xai_detector/{dataset}",
)

detector_results = xai_detector.evaluate_paths(real_path, syn_path)
detector_results

{'detector_auc': 0.9999993767928336,
 'detector_accuracy': 0.9991560587945706,
 'detector_average_precision': 0.9999993655639845,
 'n_real': 23698,
 'n_synthetic': 23698,
 'n_features_original': 30,
 'n_features_encoded': 30}

In [21]:
import model.synthesizer.ctabgan_synthesizer as ctab
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

def reconstruct_discriminator(snapshot, dside, layers_D, device):
    disc = ctab.Discriminator(dside, layers_D).to(device)
    disc.load_state_dict(snapshot["state_dict"])
    disc.eval()
    return disc

In [22]:
reconstructed_discriminators = {}

for run_name, snapshots in all_discriminator_snapshots.items():
    reconstructed_discriminators[run_name] = {}

    for snapshot in snapshots:
        epoch = snapshot["epoch"]

        disc = reconstruct_discriminator(
            snapshot=snapshot,
            dside=dside,
            layers_D=layers_D,
            device=device
        )

        reconstructed_discriminators[run_name][epoch] = disc


In [23]:
disc_run0_epoch25 = reconstructed_discriminators["run_0"][25]
disc_run1_epoch100 = reconstructed_discriminators["run_1"][100]

In [24]:
class DiscriminatorTabularWrapper(torch.nn.Module):
    def __init__(self, discriminator, Dtransformer, cond_dim=0, device="cuda"):
        super().__init__()
        self.discriminator = discriminator
        self.Dtransformer = Dtransformer
        self.cond_dim = cond_dim
        self.device = device

    def forward(self, x):
        # x = encoded tabular features only: shape [batch, data_dim]

        if self.cond_dim > 0:
            c = torch.zeros(
                x.size(0),
                self.cond_dim,
                device=x.device,
                dtype=x.dtype
            )
            x = torch.cat([x, c], dim=1)

        x_img = self.Dtransformer.transform(x)
        y, _ = self.discriminator(x_img)

        return y

In [25]:
import shap
import torch
import numpy as np
import pandas as pd

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

run_name = "run_0"
snapshot = all_discriminator_snapshots[run_name][0]

disc = reconstruct_discriminator(
    snapshot=snapshot,
    dside=dside,
    layers_D=layers_D,
    device=device
)

wrapped_disc = DiscriminatorTabularWrapper(
    discriminator=disc,
    Dtransformer=synthesizer.synthesizer.Dtransformer,
    cond_dim=synthesizer.synthesizer.cond_generator.n_opt,
    device=device
).to(device)

wrapped_disc.eval()

DiscriminatorTabularWrapper(
  (discriminator): Discriminator(
    (seq): Sequential(
      (0): Conv2d(1, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (1): LayerNorm((64, 4, 4), eps=1e-05, elementwise_affine=True)
      (2): LeakyReLU(negative_slope=0.2, inplace=True)
      (3): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (4): LayerNorm((128, 2, 2), eps=1e-05, elementwise_affine=True)
      (5): LeakyReLU(negative_slope=0.2, inplace=True)
      (6): Conv2d(128, 1, kernel_size=(2, 2), stride=(1, 1))
      (7): ReLU(inplace=True)
    )
    (seq_info): Sequential(
      (0): Conv2d(1, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (1): LayerNorm((64, 4, 4), eps=1e-05, elementwise_affine=True)
      (2): LeakyReLU(negative_slope=0.2, inplace=True)
      (3): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (4): LayerNorm((128, 2, 2), eps=1e-05, elementwise_aff

In [26]:
encoded_real = synthesizer.synthesizer.transformer.transform(
    synthesizer.data_prep.df.values
)

encoded_real = torch.tensor(encoded_real, dtype=torch.float32).to(device)

background_size = 50
explain_size = 100

background = encoded_real[
    torch.randperm(encoded_real.size(0))[:background_size]
]

to_explain = encoded_real[
    torch.randperm(encoded_real.size(0))[:explain_size]
]

In [27]:
explainer = shap.GradientExplainer(wrapped_disc, background)

shap_values = explainer.shap_values(to_explain)

if isinstance(shap_values, list):
    shap_values = shap_values[0]

shap_values = np.array(shap_values)

mean_abs_shap_encoded = np.abs(shap_values).mean(axis=0)

In [ ]:
def get_encoded_feature_slices(transformer, original_columns):
    feature_slices = {}

    start = 0
    for col_name, info in zip(original_columns, transformer.output_info):
        dim = 0

        for item in info:
            dim += item[0]

        end = start + dim
        feature_slices[col_name] = slice(start, end)
        start = end

    return feature_slices

In [ ]:
original_columns = list(synthesizer.data_prep.df.columns)

feature_slices = get_encoded_feature_slices(
    synthesizer.synthesizer.transformer,
    original_columns
)

feature_importance = {}

for feature, sl in feature_slices.items():
    feature_importance[feature] = mean_abs_shap_encoded[sl].sum()

feature_importance_df = (
    pd.DataFrame({
        "feature": list(feature_importance.keys()),
        "mean_abs_shap": list(feature_importance.values())
    })
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)

feature_importance_df

In [ ]:
all_shap_results = {}

for run_name, snapshots in all_discriminator_snapshots.items():
    all_shap_results[run_name] = {}

    for snapshot in snapshots:
        epoch = snapshot["epoch"]

        disc = reconstruct_discriminator(snapshot, dside, layers_D, device)

        wrapped_disc = DiscriminatorTabularWrapper(
            discriminator=disc,
            Dtransformer=synthesizer.synthesizer.Dtransformer,
            cond_dim=synthesizer.synthesizer.cond_generator.n_opt,
            device=device
        ).to(device)

        wrapped_disc.eval()

        explainer = shap.GradientExplainer(wrapped_disc, background)
        shap_values = explainer.shap_values(to_explain)

        if isinstance(shap_values, list):
            shap_values = shap_values[0]

        mean_abs_shap_encoded = np.abs(np.array(shap_values)).mean(axis=0)

        feature_importance = {}
        for feature, sl in feature_slices.items():
            feature_importance[feature] = mean_abs_shap_encoded[sl].sum()

        df = (
            pd.DataFrame({
                "feature": list(feature_importance.keys()),
                "mean_abs_shap": list(feature_importance.values())
            })
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True)
        )

        df = df[df["feature"] != "hospital_expire_flag"]

        all_shap_results[run_name][epoch] = df